In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm

# PARAMETERS
Z_THRESHOLD = 2  # 2σ or 3σ
START_DATE = "2018-01-01"
END_DATE = "2025-01-01"

# Get S&P 500 tickers
tickers = yf.Tickers("^GSPC").symbols if hasattr(yf.Tickers("^GSPC"), 'symbols') else None

# Or just use yfinance’s built-in list (better fallback)
sp500 = pd.read_csv("https://datahub.io/core/s-and-p-500-companies/r/constituents.csv")
tickers = sp500['Symbol'].tolist()

print(f"Loaded {len(tickers)} tickers.")

# Store results
trades = []

for ticker in tqdm(tickers, desc="Processing tickers"):
    try:
        data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
        data['Return'] = data['Close'].pct_change()

        mean = data['Return'].mean()
        std = data['Return'].std()

        # Find days with >2σ drop
        data['Signal'] = data['Return'] <= (mean - Z_THRESHOLD * std)

        # Next-day return (buy after drop)
        data['Next_Return'] = data['Return'].shift(-1)

        # Collect trades
        trade_returns = data.loc[data['Signal'], 'Next_Return'].dropna().tolist()
        for r in trade_returns:
            trades.append(r)

    except Exception as e:
        continue

# Convert to numpy array for analysis
trades = np.array(trades)

# Metrics
total_trades = len(trades)
profitable_trades = np.sum(trades > 0)
prob_profit = profitable_trades / total_trades * 100
total_profit_pct = np.mean(trades) * 100
cumulative_profit = np.prod(1 + trades) - 1

print(f"Total Trades: {total_trades}")
print(f"Probability of Profit: {prob_profit:.2f}%")
print(f"Average Profit per Trade: {total_profit_pct:.3f}%")
print(f"Cumulative Profit: {cumulative_profit * 100:.2f}%")


Loaded 503 tickers.


Processing tickers:   0%|          | 0/503 [00:00<?, ?it/s]C:\Users\Anshpreet Singh\AppData\Local\Temp\ipykernel_2220\224847858.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
Processing tickers:   0%|          | 1/503 [00:03<29:00,  3.47s/it]C:\Users\Anshpreet Singh\AppData\Local\Temp\ipykernel_2220\224847858.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
Processing tickers:   0%|          | 2/503 [00:04<17:58,  2.15s/it]C:\Users\Anshpreet Singh\AppData\Local\Temp\ipykernel_2220\224847858.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
Processing tickers:   1%|          | 3/503 [00:05<13:58,  1.68s/it]C:\Users\Anshpreet Singh\AppData\Local\Temp\ipy

Total Trades: 17794
Probability of Profit: 54.14%
Average Profit per Trade: 0.412%
Cumulative Profit: 14359104671453602681716736.00%


In [3]:
from scipy import stats

# Remove NaN values if any
trades = np.array(trades)
trades = trades[~np.isnan(trades)]

# One-sample t-test against mean = 0
t_stat, p_value = stats.ttest_1samp(trades, 0)

# Because we are testing H1: mean > 0 (one-tailed)
p_value_one_tailed = p_value / 2 if t_stat > 0 else 1 - (p_value / 2)

mean_return = np.mean(trades)
std_return = np.std(trades, ddof=1)
sharpe_ratio = mean_return / std_return * np.sqrt(252)  # annualized

print(f"Mean next-day return: {mean_return * 100:.4f}%")
print(f"T-statistic: {t_stat:.3f}")
print(f"One-tailed p-value: {p_value_one_tailed:.5f}")
print(f"Sharpe Ratio (annualized): {sharpe_ratio:.2f}")

if p_value_one_tailed < 0.05:
    print("Statistically significant: evidence that next-day returns > 0")
else:
    print("Not statistically significant: no evidence of true alpha")


Mean next-day return: 0.4123%
T-statistic: 11.735
One-tailed p-value: 0.00000
Sharpe Ratio (annualized): 1.40
Statistically significant: evidence that next-day returns > 0


In [7]:
data['Next_Return'].max()*100

11.979292181785016

In [8]:
data['Next_Return'].min()*100

-14.69555001218522

In [9]:
data['Drawdown'] = data['Next_Return'] < 0

In [13]:
data['Drawdown'].sum()

np.int64(831)

In [15]:
# Sum of all losses (in %)
losses = trades[trades < 0]
total_loss_pct = losses.sum() * 100
avg_loss = losses.mean() * 100 if len(losses) > 0 else 0

print("\n=== Loss Analysis ===")
print(f"Number of losing trades: {len(losses)}")
print(f"Total loss (sum of all negative returns): {total_loss_pct:.2f}%")
print(f"Average loss per losing trade: {avg_loss:.3f}%")



=== Loss Analysis ===
Number of losing trades: 8078
Total loss (sum of all negative returns): -23224.77%
Average loss per losing trade: -2.875%


In [16]:
# Sum of all losses (in %)
losses = trades[trades > 0]
total_loss_pct = losses.sum() * 100
avg_loss = losses.mean() * 100 if len(losses) > 0 else 0

print("\n=== Loss Analysis ===")
print(f"Number of losing trades: {len(losses)}")
print(f"Total loss (sum of all negative returns): {total_loss_pct:.2f}%")
print(f"Average loss per losing trade: {avg_loss:.3f}%")



=== Loss Analysis ===
Number of losing trades: 9633
Total loss (sum of all negative returns): 30561.63%
Average loss per losing trade: 3.173%
